# 04 - Text Splitting & Chunking Strategies

> Complete guide and in-depth Arabic documentation: [`README.md`](./README.md)

Compares `CharacterTextSplitter`, `RecursiveCharacterTextSplitter`, and token-based splitting.

## 1. Import Text Splitters

In [1]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)
from langchain_core.documents import Document

print("✅ Text splitters imported successfully!")

d:\Code\AI\rag-mastery\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Text splitters imported successfully!


## 2. Prepare Sample Corpus for Chunking

In [2]:
sample_text = """Machine Learning Basics:
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.

It focuses on developing computer programs that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Algorithms are trained using labeled examples, such as an input where the desired output is known.
2. Unsupervised Learning: Used against data that has no historical labels. The system is not told the 'right' answer.
3. Reinforcement Learning: The algorithm discovers through trial and error which actions yield the greatest rewards.

Applications of ML include recommendation engines, speech recognition, autonomous driving, and medical diagnosis."""

print(f"إجمالي عدد حروف النص الأصلي: {len(sample_text)}")

إجمالي عدد حروف النص الأصلي: 777


## 3. Method 1: Fixed-Size CharacterTextSplitter
Splits text at fixed separator boundaries.

In [3]:
char_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

char_chunks = char_splitter.split_text(sample_text)

print(f"عدد القطع الناتجة بواسطة CharacterTextSplitter: {len(char_chunks)}\n")
for i, chunk in enumerate(char_chunks, 1):
    print(f"--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk)
    print("-" * 30)

عدد القطع الناتجة بواسطة CharacterTextSplitter: 6

--- Chunk 1 (175 chars) ---
Machine Learning Basics:
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.
------------------------------
--- Chunk 2 (126 chars) ---
It focuses on developing computer programs that can access data and use it to learn for themselves.
Types of Machine Learning:
------------------------------
--- Chunk 3 (122 chars) ---
1. Supervised Learning: Algorithms are trained using labeled examples, such as an input where the desired output is known.
------------------------------
--- Chunk 4 (117 chars) ---
2. Unsupervised Learning: Used against data that has no historical labels. The system is not told the 'right' answer.
------------------------------
--- Chunk 5 (116 chars) ---
3. Reinforcement Learning: The algorithm discovers through trial and error which actions yield the greatest rewards.
------------------------

## 4. Method 2: RecursiveCharacterTextSplitter (Recommended)
Hierarchically splits on paragraphs, newlines, and spaces to preserve semantic units.

In [4]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=180,
    chunk_overlap=30,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

rec_chunks = recursive_splitter.split_text(sample_text)

print(f"عدد القطع الناتجة بواسطة RecursiveCharacterTextSplitter: {len(rec_chunks)}\n")
for i, chunk in enumerate(rec_chunks, 1):
    print(f"=== Chunk {i} ({len(chunk)} chars) ===")
    print(chunk)
    print("=" * 30)

عدد القطع الناتجة بواسطة RecursiveCharacterTextSplitter: 6

=== Chunk 1 (175 chars) ===
Machine Learning Basics:
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.
=== Chunk 2 (99 chars) ===
It focuses on developing computer programs that can access data and use it to learn for themselves.
=== Chunk 3 (149 chars) ===
Types of Machine Learning:
1. Supervised Learning: Algorithms are trained using labeled examples, such as an input where the desired output is known.
=== Chunk 4 (117 chars) ===
2. Unsupervised Learning: Used against data that has no historical labels. The system is not told the 'right' answer.
=== Chunk 5 (116 chars) ===
3. Reinforcement Learning: The algorithm discovers through trial and error which actions yield the greatest rewards.
=== Chunk 6 (113 chars) ===
Applications of ML include recommendation engines, speech recognition, autonomous driving, and medical diagnosis

## 5. Inspect Chunk Overlap Mechanism
Examines how `chunk_overlap` prevents context loss at chunk boundaries.

In [5]:
if len(rec_chunks) >= 2:
    print("نهاية Chunk 1:")
    print(f"...{rec_chunks[0][-40:]}")
    print("\nبداية Chunk 2:")
    print(f"{rec_chunks[1][:40]}...")

نهاية Chunk 1:
...nce without being explicitly programmed.

بداية Chunk 2:
It focuses on developing computer progra...


## 6. Method 3: Token-Based Splitting (TikToken)
Aligns chunk boundaries with LLM token limits rather than character counts.

In [6]:
token_splitter = TokenTextSplitter(
    chunk_size=40,
    chunk_overlap=10
)

token_chunks = token_splitter.split_text(sample_text)

print(f"عدد القطع الناتجة بواسطة TokenTextSplitter: {len(token_chunks)}\n")
for i, chunk in enumerate(token_chunks[:3], 1):
    print(f"--- Token Chunk {i} ---")
    print(chunk)
    print("-" * 30)

عدد القطع الناتجة بواسطة TokenTextSplitter: 5

--- Token Chunk 1 ---
Machine Learning Basics:
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.

It focuses on developing computer programs that can access data and
------------------------------
--- Token Chunk 2 ---
 focuses on developing computer programs that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Algorithms are trained using labeled examples,
------------------------------
--- Token Chunk 3 ---
: Algorithms are trained using labeled examples, such as an input where the desired output is known.
2. Unsupervised Learning: Used against data that has no historical labels. The system
------------------------------


## 7. Compare Splitter Characteristics

In [7]:
# إنشاء كائن Document أصلي يحمل ميتا-داتا
original_doc = Document(
    page_content=sample_text,
    metadata={"source": "ml_guide.txt", "author": "Krish", "year": 2024}
)

# تقسيم الوثيقة
split_docs = recursive_splitter.split_documents([original_doc])

print(f"عدد كائنات Document الناتجة: {len(split_docs)}")
print(f"الميتا-داتا الموروثة في Chunk 1: {split_docs[0].metadata}")
print(f"الميتا-داتا الموروثة في Chunk 2: {split_docs[1].metadata}")

عدد كائنات Document الناتجة: 6
الميتا-داتا الموروثة في Chunk 1: {'source': 'ml_guide.txt', 'author': 'Krish', 'year': 2024}
الميتا-داتا الموروثة في Chunk 2: {'source': 'ml_guide.txt', 'author': 'Krish', 'year': 2024}


## Summary
Recursive splitting with 10-20% overlap provides the best baseline for RAG retrieval.